In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PhishGuard AI — RQ3: Feature Engineering
# Fit ONCE on train.csv → save vectorizer + tokenizer (never refit in later stages)
# Output:
#   rq3_feature_engineering/models/tfidf_vectorizer.pkl
#   rq3_feature_engineering/models/dl_tokenizer.pkl
#   rq3_feature_engineering/results/feature_engineering_summary.csv
#   rq3_feature_engineering/results/experiment_config.csv
# ═══════════════════════════════════════════════════════════════════════════════

# ═══ CELL 1 — Reproducibility Block ═══
import os, random, numpy as np, torch

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

print("✓ Reproducibility block applied (SEED=42)")


# ═══ CELL 2 — Drive mount + folder tree ═══
from google.colab import drive
drive.mount("/content/drive")

BASE_DIR    = "/content/drive/MyDrive/NLP FINAL"
DATASET_DIR = os.path.join(BASE_DIR, "dataset")
STAGE_DIR   = os.path.join(BASE_DIR, "rq3_feature_engineering")

MODELS_DIR   = os.path.join(STAGE_DIR, "models")
RESULTS_DIR  = os.path.join(STAGE_DIR, "results")
LOGS_DIR     = os.path.join(STAGE_DIR, "logs")
DIAGRAMS_DIR = os.path.join(STAGE_DIR, "diagrams")

for folder in [MODELS_DIR, RESULTS_DIR, LOGS_DIR, DIAGRAMS_DIR]:
    os.makedirs(folder, exist_ok=True)

TRAIN_PATH = os.path.join(DATASET_DIR, "train.csv")
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"train.csv not found at {TRAIN_PATH}\nRun Stage 0 first.")

TFIDF_PATH = os.path.join(MODELS_DIR, "tfidf_vectorizer.pkl")
DLTOK_PATH = os.path.join(MODELS_DIR, "dl_tokenizer.pkl")

print(f"✓ BASE_DIR   : {BASE_DIR}")
print(f"✓ TRAIN_PATH : {TRAIN_PATH}")
print(f"✓ MODELS_DIR : {MODELS_DIR}")


# ═══ CELL 3 — Install dependencies ═══
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "scikit-learn", "joblib", "tensorflow"])

import pandas as pd
import joblib
import platform
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer


# ═══ CELL 4 — Load train.csv ONLY (never val/test for fitting) ═══
train_df = pd.read_csv(TRAIN_PATH)

required = ["text_cleaned_classical", "text_cleaned_transformer", "label"]
missing  = [c for c in required if c not in train_df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

classical_texts   = train_df["text_cleaned_classical"].astype(str).tolist()
transformer_texts = train_df["text_cleaned_transformer"].astype(str).tolist()

print(f"Loaded train.csv → {len(train_df):,} samples (fit scope = TRAIN ONLY)")


# ═══ CELL 5 — TF-IDF Vectorizer (classical pipeline → RQ4) ═══
print("\n[1/2] Fitting TF-IDF vectorizer on text_cleaned_classical ...")

tfidf_vectorizer = TfidfVectorizer(
    max_features=10_000,       # cap vocabulary at 10k most frequent uni+bigrams
    ngram_range=(1, 2),        # unigrams + bigrams
    min_df=2,                  # ignore terms appearing in < 2 docs (reduces noise)
    max_df=0.95,               # ignore terms in > 95% of docs (removes near-stopwords)
    sublinear_tf=True,         # log-scaled TF — standard for email/text classification
    dtype=np.float32,
)

X_tfidf_train = tfidf_vectorizer.fit_transform(classical_texts)

actual_tfidf_features = X_tfidf_train.shape[1]
tfidf_vocab_size      = len(tfidf_vectorizer.vocabulary_)

print(f"  TF-IDF matrix shape (train) : {X_tfidf_train.shape}")
print(f"  Vocabulary entries stored   : {tfidf_vocab_size:,}")
print(f"  Feature dimensionality      : {actual_tfidf_features:,}")
print(f"  Sparsity                    : {100 * (1 - X_tfidf_train.nnz / (X_tfidf_train.shape[0] * X_tfidf_train.shape[1])):.2f}%")

joblib.dump(tfidf_vectorizer, TFIDF_PATH)
print(f"  ✓ Saved → {TFIDF_PATH}")


# ═══ CELL 6 — DL Tokenizer (transformer-style text → RQ5 BiLSTM / TextCNN) ═══
print("\n[2/2] Fitting DL tokenizer on text_cleaned_transformer ...")

dl_tokenizer = Tokenizer(
    oov_token="<OOV>",
    lower=False,    # preserve casing — matches transformer-style column intent
    filters="",     # no extra filtering — text already cleaned in Stage 0
)

dl_tokenizer.fit_on_texts(transformer_texts)

dl_vocab_size = len(dl_tokenizer.word_index) + 1   # +1 for padding index 0

# Sequence length stats (for RQ5 padding reference)
seq_lengths = [len(seq) for seq in dl_tokenizer.texts_to_sequences(transformer_texts)]
p50_len = int(np.percentile(seq_lengths, 50))
p95_len = int(np.percentile(seq_lengths, 95))
p99_len = int(np.percentile(seq_lengths, 99))
max_len = int(np.max(seq_lengths))

print(f"  DL vocabulary size          : {dl_vocab_size:,}")
print(f"  Sequence length — median/p95/max : {p50_len} / {p95_len} / {max_len}")
print(f"  Recommended pad_length (p95)  : {p95_len}")

joblib.dump(dl_tokenizer, DLTOK_PATH)
print(f"  ✓ Saved → {DLTOK_PATH}")


# ═══ CELL 7 — Verify saved artifacts reload correctly ═══
print("\nVerifying reload ...")

tfidf_loaded = joblib.load(TFIDF_PATH)
dltok_loaded = joblib.load(DLTOK_PATH)

assert tfidf_loaded.transform(classical_texts[:5]).shape[1] == actual_tfidf_features
assert len(dltok_loaded.word_index) + 1 == dl_vocab_size

print("  ✓ tfidf_vectorizer.pkl reload OK")
print("  ✓ dl_tokenizer.pkl reload OK")


# ═══ CELL 8 — feature_engineering_summary.csv ═══
print("\nBuilding feature_engineering_summary.csv ...")

# Top TF-IDF features sample (for report reference)
feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_means   = np.asarray(X_tfidf_train.mean(axis=0)).flatten()
top_tfidf_idx = tfidf_means.argsort()[-10:][::-1]
top_tfidf_feats = ", ".join([feature_names[i] for i in top_tfidf_idx])

# Top DL tokens sample
top_dl_tokens = ", ".join(
    [w for w, _ in sorted(dl_tokenizer.word_counts.items(), key=lambda x: -x[1])[:10]]
)

summary_rows = [
    {
        "artifact": "tfidf_vectorizer.pkl",
        "feature_type": "TF-IDF (sparse matrix)",
        "source_column": "text_cleaned_classical",
        "used_by_stages": "RQ4 (NB, LogReg, SVM, RF, XGBoost) | RQ7 tuning | RQ9 LIME | RQ10 app",
        "fit_scope": "train.csv ONLY",
        "max_features_cap": 10_000,
        "ngram_range": "(1, 2)",
        "vocab_size": tfidf_vocab_size,
        "feature_dimensionality": actual_tfidf_features,
        "recommended_pad_length": "N/A",
        "sequence_length_p95": "N/A",
        "top_features_sample": top_tfidf_feats,
        "justification": (
            "TF-IDF with uni+bigrams captures single phishing cue words ('verify', 'password') "
            "and two-word phrases ('click here', 'account suspended'). max_features=10000 balances "
            "expressiveness vs. memory. min_df=2 and max_df=0.95 remove ultra-rare and ultra-common "
            "noise. sublinear_tf dampens the impact of very long emails. Fit once on train — "
            "RQ4/RQ7/RQ9/RQ10 load this file, never refit."
        ),
    },
    {
        "artifact": "dl_tokenizer.pkl",
        "feature_type": "Keras Tokenizer (integer sequences)",
        "source_column": "text_cleaned_transformer",
        "used_by_stages": "RQ5 (BiLSTM, TextCNN)",
        "fit_scope": "train.csv ONLY",
        "max_features_cap": "None (full train vocabulary)",
        "ngram_range": "word-level (whitespace split)",
        "vocab_size": dl_vocab_size,
        "feature_dimensionality": dl_vocab_size,
        "recommended_pad_length": p95_len,
        "sequence_length_p95": p95_len,
        "top_features_sample": top_dl_tokens,
        "justification": (
            "Keras Tokenizer on transformer-style text (case + punctuation preserved) for "
            "BiLSTM and TextCNN in RQ5. lower=False and filters='' respect the dual-pipeline "
            "design — DL models need natural token boundaries, not heavy normalization. "
            "OOV token handles unseen words at inference. DistilBERT (RQ6) uses its own "
            "HuggingFace WordPiece tokenizer — NOT this file. Fit once on train — RQ5 loads "
            f"this file, never refit. Use pad_length={p95_len} (95th percentile) in RQ5."
        ),
    },
    {
        "artifact": "distilbert_tokenizer",
        "feature_type": "HuggingFace AutoTokenizer (WordPiece)",
        "source_column": "text_cleaned_transformer",
        "used_by_stages": "RQ6 (DistilBERT) — separate from dl_tokenizer.pkl",
        "fit_scope": "Pretrained (distilbert-base-uncased) — NOT fit on this dataset",
        "max_features_cap": "30522 (pretrained vocab)",
        "ngram_range": "subword (WordPiece)",
        "vocab_size": 30_522,
        "feature_dimensionality": 768,
        "recommended_pad_length": 512,
        "sequence_length_p95": p99_len,
        "top_features_sample": "N/A — pretrained",
        "justification": (
            "DistilBERT uses a pretrained subword tokenizer loaded from HuggingFace in RQ6. "
            "It is documented here for completeness but is NOT saved in rq3 — RQ6 loads "
            "distilbert-base-uncased directly. max_length=512 (BERT standard)."
        ),
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_path = os.path.join(RESULTS_DIR, "feature_engineering_summary.csv")
summary_df.to_csv(summary_path, index=False)
print(f"  ✓ Saved → {summary_path}")


# ═══ CELL 9 — Paper-ready table ═══
paper_table = summary_df[[
    "artifact", "source_column", "used_by_stages",
    "vocab_size", "feature_dimensionality", "recommended_pad_length", "justification"
]].copy()
paper_table.columns = [
    "Artifact", "Source Column", "Used By",
    "Vocab Size", "Feature Dim", "Pad Length", "Justification"
]
paper_path = os.path.join(RESULTS_DIR, "paper_table_rq3_feature_engineering.csv")
paper_table.to_csv(paper_path, index=False)
print(f"  ✓ Saved → {paper_path}")


# ═══ CELL 10 — experiment_config.csv ═══
config = {
    "project":                    "PhishGuard AI",
    "stage":                      "RQ3 — Feature Engineering",
    "timestamp_utc":              datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),
    "seed":                       SEED,
    "fit_scope":                  "train.csv ONLY (never val/test)",
    "train_samples":              len(train_df),
    "tfidf_source_column":        "text_cleaned_classical",
    "tfidf_max_features":         10_000,
    "tfidf_ngram_range":          "(1, 2)",
    "tfidf_min_df":               2,
    "tfidf_max_df":               0.95,
    "tfidf_sublinear_tf":         True,
    "tfidf_vocab_size":           tfidf_vocab_size,
    "tfidf_feature_dimensionality": actual_tfidf_features,
    "tfidf_save_path":            TFIDF_PATH,
    "dl_source_column":           "text_cleaned_transformer",
    "dl_tokenizer_type":          "Keras Tokenizer",
    "dl_lower":                   False,
    "dl_filters":                 "empty (none)",
    "dl_oov_token":               "<OOV>",
    "dl_vocab_size":              dl_vocab_size,
    "dl_seq_length_median":       p50_len,
    "dl_seq_length_p95":          p95_len,
    "dl_seq_length_max":          max_len,
    "dl_recommended_pad_length":  p95_len,
    "dl_save_path":               DLTOK_PATH,
    "never_refit_rule":           "RQ4-RQ10 must load these pickles — never refit",
    "python_version":             platform.python_version(),
    "pandas_version":             pd.__version__,
    "numpy_version":              np.__version__,
    "sklearn_version":            __import__("sklearn").__version__,
    "tensorflow_version":         __import__("tensorflow").__version__,
}
config_df = pd.DataFrame(list(config.items()), columns=["parameter", "value"])
config_path = os.path.join(RESULTS_DIR, "experiment_config.csv")
config_df.to_csv(config_path, index=False)
print(f"  ✓ Saved → {config_path}")


# ═══ CELL 11 — Final summary ═══
print("\n" + "=" * 70)
print("RQ3 — FEATURE ENGINEERING COMPLETE ✓")
print("=" * 70)
print(f"\nFit scope : train.csv only ({len(train_df):,} samples)")
print(f"\nSaved models:")
print(f"  tfidf_vectorizer.pkl  →  {actual_tfidf_features:,} features  (uni+bigram TF-IDF)")
print(f"  dl_tokenizer.pkl      →  {dl_vocab_size:,} vocab tokens  (Keras, lower=False)")
print(f"\nRQ5 padding hint : pad_length = {p95_len}  (95th percentile word count)")
print(f"\nResults → {RESULTS_DIR}/")
print("  feature_engineering_summary.csv")
print("  paper_table_rq3_feature_engineering.csv")
print("  experiment_config.csv")
print("\nNext step → RQ4 (Classical ML: NB, LogReg, SVM, RF, XGBoost)")
print("  RQ4 loads: tfidf_vectorizer.pkl + train/val/test.csv")
print("=" * 70)

display(paper_table)

✓ Reproducibility block applied (SEED=42)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ BASE_DIR   : /content/drive/MyDrive/NLP FINAL
✓ TRAIN_PATH : /content/drive/MyDrive/NLP FINAL/dataset/train.csv
✓ MODELS_DIR : /content/drive/MyDrive/NLP FINAL/rq3_feature_engineering/models
Loaded train.csv → 13,041 samples (fit scope = TRAIN ONLY)

[1/2] Fitting TF-IDF vectorizer on text_cleaned_classical ...
  TF-IDF matrix shape (train) : (13041, 10000)
  Vocabulary entries stored   : 10,000
  Feature dimensionality      : 10,000
  Sparsity                    : 99.08%
  ✓ Saved → /content/drive/MyDrive/NLP FINAL/rq3_feature_engineering/models/tfidf_vectorizer.pkl

[2/2] Fitting DL tokenizer on text_cleaned_transformer ...
  DL vocabulary size          : 203,971
  Sequence length — median/p95/max : 160 / 1234 / 24062
  Recommended pad_length (p95)  : 1234
  ✓ Saved → /content/drive/MyDrive/NLP FINAL/rq3_feature_e

/tmp/ipykernel_1069/886957803.py:255: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp_utc":              datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S"),


,Artifact,Source Column,Used By,Vocab Size,Feature Dim,Pad Length,Justification
0,tfidf_vectorizer.pkl,text_cleaned_classical,"RQ4 (NB, LogReg, SVM, RF, XGBoost) | RQ7 tunin...",10000,10000,N/A,TF-IDF with uni+bigrams captures single phishi...
1,dl_tokenizer.pkl,text_cleaned_transformer,"RQ5 (BiLSTM, TextCNN)",203971,203971,1234,Keras Tokenizer on transformer-style text (cas...
2,distilbert_tokenizer,text_cleaned_transformer,RQ6 (DistilBERT) — separate from dl_tokenizer.pkl,30522,768,512,DistilBERT uses a pretrained subword tokenizer...
